# ArabicDeXlit — Train Your Own Arabic ASR De-Transliterator

> **Undo Arabic ASR transliteration:** turn `انترن` back into `intern`, `ايه اي` back into `AI` —
> while leaving genuinely Arabic text **byte-identical**.

Arabic ASR models transcribe spoken English using Arabic letters. This notebook trains a small
two-stage model that repairs that, as a drop-in post-processor.

| | |
|---|---|
| **Input** | `كنت انترن في او اي اي و بعدها جالي اوفر` |
| **Output** | `كنت intern في OIE و بعدها جالي offer` |
| **Pure Arabic in** | `أنا رايح البيت دلوقتي` |
| **Pure Arabic out** | `أنا رايح البيت دلوقتي` *(unchanged, guaranteed)* |

### How it works

**Stage 1** *detects* which tokens are transliterated (a token tagger).
**Stage 2** *converts* each span through a **typed hybrid router** -- deterministic
where structure exists, neural where ambiguity exists:

| Category | Path |
|---|---|
| `EMAIL` `URL` `NUMBER` `TIME` | parser (exact, or declines) |
| `ACRONYM` | closed inventory lookup |
| `ENTITY` | gazetteer, then neural |
| `CS` | neural + rule prior + sentence context |

Every path can decline, and a span nothing is confident about is **copied unchanged**.

Tokens tagged `O` are copied verbatim, so a fully-Arabic sentence is returned unchanged
**by construction** — not because the model learned to behave. A sentence with no code-switching
never reaches stage 2 at all, which is also why latency stays near zero.

### What you need

- **Runtime → Change runtime type → GPU.** T4, L4 or A100 all work.
- A [Weights & Biases](https://wandb.ai) account (optional, for monitoring).
- Nothing else — the dataset is downloaded ready-built from the Hub.

| GPU | Config used | Approx. time |
|---|---|---|
| A100 | `detector_base.yaml` | ~1–1.5 h |
| L4 | `detector_base.yaml` | ~2–3 h |
| T4 | `detector_t4.yaml` | ~3–4 h |

Stage 2 adds only a few minutes on any of them.

## 1. Check the GPU

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)[0]
    print("GPU:", torch.cuda.get_device_name(0))
    # Ampere and newer (A100, L4) have real bf16; T4 must use fp16. The trainer
    # picks this automatically -- shown here so you know what you're getting.
    print("autocast dtype:", "bfloat16" if cap >= 8 else "float16")
else:
    print("No GPU detected. Runtime -> Change runtime type -> GPU")

## 2. Get the code

In [ ]:
import os

if not os.path.exists("Arabic-DeXlit"):
    !git clone -q https://github.com/MohammedAly22/Arabic-DeXlit.git
%cd Arabic-DeXlit
!pip install -q -r requirements.txt

print("\nready")

## 3. Load the dataset from the Hugging Face Hub

The corpus is already built and published, so there is nothing to generate here — this pulls
~68 MB of Parquet and writes the JSONL splits the training scripts read.

**[`mohammedaly22/ArabicDeXlit-Corpus`]( https://huggingface.co/datasets/mohammedaly22/ArabicDeXlit-Corpus)**

| | |
|---|---|
| Examples | 475,945 |
| Dialects | 8 (MSA, Egyptian, Gulf, Levantine, Iraqi, Maghrebi, Sudanese, Yemeni) |
| Pass-through rows | 30% — pure Arabic, where the correct answer is *change nothing* |
| Leakage | none: 0 overlapping sentence families between train / validation / test |

In [ ]:
import sys
sys.path.insert(0, "src")
from arabic_dexlit.data.hub import download_corpus, corpus_summary

download_corpus("data/processed", repo_id="mohammedaly22/ArabicDeXlit-Corpus")

import json
print(json.dumps(corpus_summary("data/processed"), indent=2))

In [ ]:
# A look at what the model actually learns from.
import json

rows = [json.loads(l) for l in open("data/processed/validation.jsonl", encoding="utf-8").readlines()[:400]]

conv = next(r for r in rows if r["spans"])
print("--- a conversion example ---")
print("INPUT :", conv["src"])
print("TARGET:", conv["tgt"])
print("TAGS  :", list(zip(conv["src_tokens"], conv["tags"])))

keep = next(r for r in rows if all(t == "O" for t in r["tags"]))
print("\n--- a pass-through example (must not change) ---")
print("INPUT :", keep["src"])
print("TARGET:", keep["tgt"])
print("identical:", keep["src"] == keep["tgt"])

<details>
<summary><b>Optional: rebuild the dataset from scratch instead</b></summary>

You only need this if you want to change how the data is made — a different pass-through ratio,
more augmentation variants, or extra dialect synthesis. It downloads the source corpora and
regenerates everything (10–20 minutes).

```python
!python scripts/build_dataset.py --fetch-mono 150000 --passthrough-ratio 0.30 --variants 1
```

To add more dialect and acronym/email coverage with Gemini
(free key from [Google AI Studio](https://aistudio.google.com/apikey)):

```python
import os
os.environ["GEMINI_API_KEY"] = "..."
!python scripts/synthesize_data.py --total 30000 --workers 16
!python scripts/build_dataset.py --passthrough-ratio 0.30
```
</details>

### The deterministic paths need no model

These run before the neural converter and are exact when they apply, so they cannot be
second-guessed by a checkpoint. Worth seeing before training starts.

In [ ]:
from arabic_dexlit.convert.parsers import parse_span
from arabic_dexlit.convert.translit_rules import arabic_to_latin

print("parsers (exact, or they decline):")
for text, cat in [
    ("تو زيرو تو فور", "NUMBER"),
    ("احمد ات جيميل دوت كوم", "EMAIL"),
    ("دبليو دبليو دبليو دوت جوجل دوت كوم", "EMAIL"),
    ("كلام عربي عادي", "EMAIL"),
]:
    print(f"  [{cat:7}] {text:36} -> {parse_span(text.split(), cat)!r}")

print()
print("rule transliteration is a PRIOR, not an answer:")
for w, gold in [("فريندس", "friends"), ("ميتينج", "meeting"), ("انترن", "intern")]:
    print(f"  {w:10} -> {arabic_to_latin(w):10} (gold: {gold})")
print()
print("-> the neural model corrects these skeletons instead of deriving them")

## 4. Weights & Biases (optional)

Tracks loss, accuracy, the safety metrics and the diagnostic plots. Skip this cell to train without logging.

In [ ]:
WANDB_PROJECT = "arabic-dexlit"   # set to None to disable

if WANDB_PROJECT:
    import wandb
    wandb.login()   # paste your key from https://wandb.ai/authorize

## 5. Smoke test

Runs the whole pipeline on a few hundred examples in about a minute.

**Always run this before starting a long GPU session.** It catches a broken path immediately
rather than forty minutes in.

In [ ]:
!python scripts/train.py --stage both --smoke --no-wandb

## 6. Train stage 1 — the span detector

This is the main run. The config is chosen automatically from your GPU.

Watch these two metrics above all others:

- **`val/passthrough_accuracy`** — of the sentences that must be returned untouched, how many were.
  This is the safety metric. Aim for ≥ 0.99.
- **`val/false_edit_rate`** — the share of ordinary tokens the model wanted to edit. Lower is better.

A model with a great F1 but a poor pass-through score is **not usable**: it corrupts normal Arabic.
Checkpoints are selected on `span_f1 - 0.5 × false_edit_rate` for exactly that reason.

In [ ]:
import torch
cap = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
CONFIG = "configs/detector_base.yaml" if cap >= 8 else "configs/detector_t4.yaml"
print("using", CONFIG)

wandb_arg = f"--wandb-project {WANDB_PROJECT}" if WANDB_PROJECT else "--no-wandb"

# Built as one string and run with a single {} interpolation: expanding a
# multi-word $variable across a line-continued ! command is fragile.
cmd = (
    f"python scripts/train.py --stage detector "
    f"--config {CONFIG} --output-dir outputs/dexlit {wandb_arg}"
)
print(cmd)
!{cmd}

## 7. Train stage 2 — the span converter

Small and fast: a few million parameters over short span pairs, usually just minutes. It is saved
into the same directory as the detector so the pipeline loads both from one path.

In [ ]:
cmd = (
    "python scripts/train.py --stage converter "
    "--converter-config configs/converter_base.yaml "
    f"--output-dir outputs/dexlit {wandb_arg}"
)
print(cmd)
!{cmd}

## 8. Try it

The first block is the real test: these must come back **character for character identical**.

In [ ]:
from arabic_dexlit.inference.pipeline import DeXlitPipeline

pipe = DeXlitPipeline.from_pretrained("outputs/dexlit", device="cuda")

print("=== pure Arabic: must be returned UNCHANGED ===")
pure = [
    "أنا رايح البيت دلوقتي عشان تعبان جدا",
    "الحمد لله على كل حال يا صديقي",
    "شلونك اليوم؟ ان شاء الله بخير",
    "كيفك؟ شو اخبارك اليوم",
]
ok = 0
for s in pure:
    out = pipe.predict(s)
    good = out.text == s
    ok += good
    print(("PASS" if good else "FAIL"), "|", out.text)
print(f"\n{ok}/{len(pure)} returned byte-identical")

In [ ]:
print("=== transliterated input: should be converted ===")
tests = [
    "كنت انترن في او اي اي اللي هي اورانج انوفيشن ايجيبت و بعدها جالي اوفر",
    "و هنا كانت بداية تعاملي مع ال سبيتش ايه اي و ال ارابيك ان ال بي",
    "ابعتلي على احمد ات جيميل دوت كوم",
    "كنت شغال ريموتلي و بقبض كويس",
    "عندي ميتينج مع ال مانجر بكرة",
]
for s in tests:
    out = pipe.predict(s)
    print("IN  :", s)
    print("OUT :", out.text)
    for x in out.spans:
        print(f"      [{x['category']}] {x['original']} -> {x['converted']}")
    for p in out.protected:
        print(f"      [PROTECTED:{p['reason']}] {p['token']}")
    print()

## 9. Evaluate on the held-out test set

In [ ]:
!python scripts/evaluate.py \
    --model-dir outputs/dexlit \
    --data data/processed/test.jsonl \
    --out outputs/dexlit/test_report.json

## 10. Save your work

**Colab wipes the VM when the session ends.** Run at least one of these.

In [ ]:
# Option A -- copy to Google Drive
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/ArabicDeXlit
!cp -r outputs/dexlit /content/drive/MyDrive/ArabicDeXlit/
print("saved to Drive")

In [ ]:
# Option B -- push the trained model to the Hugging Face Hub
from huggingface_hub import notebook_login, HfApi

notebook_login()

REPO = "mohammedaly22/ArabicDeXlit-base"
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="outputs/dexlit", repo_id=REPO,
                  commit_message="Add trained ArabicDeXlit model")
print(f"https://huggingface.co/{REPO}")

---

## Troubleshooting

**Out of memory.** Lower `--batch-size` (try 16, then 8) and raise `--grad-accum` to keep the
effective batch size the same. On a T4, `configs/detector_t4.yaml` already freezes the bottom 6
encoder layers, which is the single biggest saving.

**Training is too slow.** Cap the corpus with `--max-train-examples 150000`, or raise
`--freeze-encoder-layers`. Stage 1 dominates the runtime; stage 2 is always quick.

**Pass-through accuracy is low.** The model is editing text it should leave alone. Rebuild with a
higher `--passthrough-ratio` (0.4), or raise `false_edit_penalty` in the config so checkpoint
selection punishes false edits harder.

**The model misses acronyms or emails.** Those categories are rarer in the corpus than plain
code-switching. Generate more with `scripts/synthesize_data.py --categories ACRONYM EMAIL`.

**Colab disconnected mid-run.** Re-run sections 1–3, then 6. Training restarts from scratch, so on
long runs save checkpoints to Drive (section 10) as you go.

---

- **Code:** <https://github.com/MohammedAly22/Arabic-DeXlit>
- **Dataset:** <https://huggingface.co/datasets/mohammedaly22/ArabicDeXlit-Corpus>
- **Model:** <https://huggingface.co/mohammedaly22/ArabicDeXlit-base>